# IMPORTS

In [62]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, cross_val_score, StratifiedKFold

In [19]:
import os
from google.colab import drive
drive.mount('/content/gdrive')

# Change working directory to be current folder
# os.chdir('/content/gdrive/My Drive/Your Folder Name/Your sub Folder Name')
os.chdir('/content/gdrive/My Drive/Projects_IAN/iris_production')
!ls

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
iris.csv  iris_to_csv.ipynb  iris_training_explore.ipynb


# Preprocess, Training and Test Set

In [20]:
df = pd.read_csv("iris.csv")

In [21]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2,virginica
146,6.3,2.5,5.0,1.9,2,virginica
147,6.5,3.0,5.2,2.0,2,virginica
148,6.2,3.4,5.4,2.3,2,virginica


In [22]:
X, y = df.iloc[:, :-2], df.iloc[:, -2]

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression

In [54]:
log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

# Evaluation on Simple Train Test

In [55]:
y_pred = log_reg_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

In [56]:
print(accuracy)
print(classification_report(y_test, y_pred))

1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [57]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]


# Evaluation and Fit with K_Fold


In [64]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

# scores = cross_val_score(log_reg_model, X, y, cv=skf)

scores = cross_validate(
    log_reg_model,
    X,
    y,
    cv=skf,
    scoring=metrics
)

# print(scores.mean())
print(scores)


{'fit_time': array([0.25684047, 0.1232574 , 0.05156446, 0.06555724, 0.08676481]), 'score_time': array([0.04077268, 0.01247263, 0.01229477, 0.01249242, 0.01248908]), 'test_accuracy': array([1.        , 0.96666667, 0.93333333, 1.        , 0.93333333]), 'test_precision_macro': array([1.        , 0.96969697, 0.94444444, 1.        , 0.93333333]), 'test_recall_macro': array([1.        , 0.96666667, 0.93333333, 1.        , 0.93333333]), 'test_f1_macro': array([1.        , 0.96658312, 0.93265993, 1.        , 0.93333333])}


In [65]:
accuracy = scores["test_accuracy"]
precision = scores["test_precision_macro"]
recall = scores["test_recall_macro"]
f1 = scores["test_f1_macro"]

In [72]:
print(f"accuracy =    {accuracy}")
print(f"percision =   {precision}")
print(f"recall =      {recall}")
print(f"f1 =          {f1}")

accuracy =    [1.         0.96666667 0.93333333 1.         0.93333333]
percision =   [1.         0.96969697 0.94444444 1.         0.93333333]
recall =      [1.         0.96666667 0.93333333 1.         0.93333333]
f1 =          [1.         0.96658312 0.93265993 1.         0.93333333]


In [67]:
print(type(scores))

<class 'dict'>


# Evaluation and Fit with K_Fold (explicit way)


In [50]:
scores = []

for train_index, test_index in skf.split(X,y):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    log_reg_model.fit(X_train, y_train)

    preds = log_reg_model.predict(X_test)

    score = accuracy_score(y_test, preds)

    scores.append(score)

print(np.mean(scores)) # scores here is list not numpy array
print(scores)

[1.0, 0.9666666666666667, 0.9333333333333333, 1.0, 0.9333333333333333]
0.9666666666666668


# Testing Syntax


In [58]:
[y_test.iloc[0:3]]

[73     1
 18     0
 118    2
 Name: target, dtype: int64]

In [59]:
log_reg_model.predict(X_test.iloc[[0]])

array([1])

In [60]:
log_reg_model.predict_proba(X_test.iloc[[0]])[0][log_reg_model.predict(X_test.iloc[[0]])]

array([0.82773966])

In [61]:
probs = log_reg_model.predict_proba(X_test)
print(probs[:3])

[[3.80392750e-03 8.27739664e-01 1.68456409e-01]
 [9.46973015e-01 5.30267859e-02 1.98766047e-07]
 [8.86161500e-09 1.54855320e-03 9.98451438e-01]]
